# 06b — E3b Gradio Deepfake-Detection Application

This notebook implements a video-analysis research demonstrator using the retained **E3b official-SBI EfficientNet-B0 checkpoint**. A user uploads one video, ten frames are sampled across its duration, the largest detected face is cropped with a 15% margin, and E3b produces frame-level fake scores. Their arithmetic mean forms the video score.

Two fixed operating points are reported:

- **Standard mode:** $\tau=0.50$, retained from the canonical evaluation.
- **High-sensitivity mode:** $\tau=0.17$, selected using video-level macro F1 on genuine FaceForensics++ validation videos in notebook 06a.

The interactive slider is educational. Slider values other than 0.50 and 0.17 have not been validated and must not be presented as optimised operating points.

## 1. Install dependencies and mount Google Drive

In [2]:
INSTALL_DEPENDENCIES = True
CREATE_PUBLIC_LINK = True

from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "gradio", "retina-face", "tf-keras",
    ])

print("Dependencies are ready.")

Mounted at /content/drive
Dependencies are ready.


## 2. Imports, fixed operating points and model path

The high-sensitivity threshold is deliberately hard-coded. It must not be changed after examining test or uploaded-video outcomes.

In [3]:
from pathlib import Path

import cv2
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from retinaface import RetinaFace
from torchvision.models import efficientnet_b0

BASE_PATH = Path("/content/drive/MyDrive/deepfake_project")
CHECKPOINT_PATH = BASE_PATH / "saved_models" / "efficientnet_b0_official_sbi_best.pth"

# Canonical threshold retained from the original E3b evaluation.
STANDARD_THRESHOLD = 0.50

# Selected only from genuine FF++ validation videos in notebook 06a.
# No FF++ test or Celeb-DF labels were used to select this value.
SENSITIVE_THRESHOLD = 0.17

NUMBER_OF_SAMPLED_FRAMES = 10
FACE_MARGIN_RATIO = 0.15
IMAGE_SIZE = 224
REAL_INDEX = 0
FAKE_INDEX = 1

assert STANDARD_THRESHOLD == 0.50
assert SENSITIVE_THRESHOLD == 0.17
assert CHECKPOINT_PATH.is_file(), f"E3b checkpoint not found: {CHECKPOINT_PATH}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH)
print("Standard threshold:", STANDARD_THRESHOLD)
print("High-sensitivity threshold:", SENSITIVE_THRESHOLD)

Device: cuda
Checkpoint: /content/drive/MyDrive/deepfake_project/saved_models/efficientnet_b0_official_sbi_best.pth
Standard threshold: 0.5
High-sensitivity threshold: 0.17


## 3. Strictly rebuild and load E3b

In [4]:
def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state = checkpoint["state_dict"]
    else:
        state = checkpoint

    if not isinstance(state, dict):
        raise TypeError("Checkpoint does not contain a state dictionary.")

    return {key.removeprefix("module."): value for key, value in state.items()}


model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(extract_state_dict(checkpoint), strict=True)

if isinstance(checkpoint, dict):
    recorded_classes = checkpoint.get("classes")
    if recorded_classes not in (None, ["real", "fake"]):
        raise RuntimeError(f"Unexpected checkpoint class order: {recorded_classes}")

model = model.to(DEVICE)
model.eval()

print("PASS: retained E3b checkpoint loaded with strict state-dictionary matching.")

PASS: retained E3b checkpoint loaded with strict state-dictionary matching.


## 4. Video sampling, RetinaFace cropping and E3b preprocessing

In [5]:
def sample_video_frames(video_path, number_of_frames=NUMBER_OF_SAMPLED_FRAMES):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError("The uploaded video could not be opened.")

    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        capture.release()
        raise ValueError("The uploaded video does not report a valid frame count.")

    sample_count = min(number_of_frames, total_frames)
    frame_indices = np.linspace(0, total_frames - 1, sample_count, dtype=int)
    sampled = []

    for sample_number, frame_index in enumerate(frame_indices, start=1):
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame_bgr = capture.read()
        sampled.append({
            "sample_number": sample_number,
            "frame_index": int(frame_index),
            "read_success": bool(success),
            "frame_bgr": frame_bgr if success else None,
        })

    capture.release()
    return sampled, total_frames


def crop_largest_face(frame_bgr):
    image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    try:
        detections = RetinaFace.detect_faces(image_rgb)
    except Exception:
        return None

    if not isinstance(detections, dict):
        return None

    best_box = None
    best_area = 0

    for detection in detections.values():
        x1, y1, x2, y2 = detection["facial_area"]
        area = max(0, x2 - x1) * max(0, y2 - y1)
        if area > best_area:
            best_area = area
            best_box = (x1, y1, x2, y2)

    if best_box is None:
        return None

    x1, y1, x2, y2 = best_box
    height, width = frame_bgr.shape[:2]
    face_width = x2 - x1
    face_height = y2 - y1
    margin_x = int(face_width * FACE_MARGIN_RATIO)
    margin_y = int(face_height * FACE_MARGIN_RATIO)

    x1 = max(int(x1 - margin_x), 0)
    y1 = max(int(y1 - margin_y), 0)
    x2 = min(int(x2 + margin_x), width)
    y2 = min(int(y2 + margin_y), height)

    crop_bgr = frame_bgr[y1:y2, x1:x2]
    if crop_bgr.size == 0:
        return None

    return cv2.resize(crop_bgr, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)


def e3b_tensor(crop_bgr):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    image = crop_rgb.astype(np.float32) / 255.0
    image = image.transpose(2, 0, 1)
    return torch.tensor(image, dtype=torch.float32)


def predict_fake_scores(crops_bgr):
    batch = torch.stack([e3b_tensor(crop) for crop in crops_bgr]).to(DEVICE)
    with torch.no_grad():
        probabilities = torch.softmax(model(batch), dim=1)[:, FAKE_INDEX]
    return probabilities.cpu().numpy().astype(float).tolist()

## 5. Application analysis and display helpers

In [6]:
def label_for_score(score, threshold):
    return "Potential manipulation" if score >= threshold else "Authentic"


def result_badge(label):
    if label == "Potential manipulation":
        return "🔴 **Potential manipulation**"
    return "🟢 **Authentic**"


def explorer_message(score, threshold):
    if score is None or not np.isfinite(score):
        return "Upload and analyse a video before using the threshold explorer."

    label = label_for_score(float(score), float(threshold))
    validation_note = (
        "Validated operating point."
        if np.isclose(threshold, STANDARD_THRESHOLD) or np.isclose(threshold, SENSITIVE_THRESHOLD)
        else "Exploratory threshold; this value was not selected by the validation protocol."
    )
    return (
        f"### Exploratory decision: {result_badge(label)}\n"
        f"Video score: **{score:.4f}**  |  Threshold: **{threshold:.2f}**  \n"
        f"{validation_note}"
    )


def empty_plot(message):
    figure, axis = plt.subplots(figsize=(8, 4))
    axis.text(0.5, 0.5, message, ha="center", va="center", transform=axis.transAxes)
    axis.set_axis_off()
    figure.tight_layout()
    return figure


def analyse_video(video_path, exploratory_threshold):
    if not video_path:
        raise gr.Error("Select a video before starting the analysis.")

    try:
        sampled_frames, total_frames = sample_video_frames(video_path)
    except Exception as error:
        raise gr.Error(str(error)) from error

    rows = []
    valid_crops = []
    valid_row_indices = []
    gallery_items = []

    for row_index, sample in enumerate(sampled_frames):
        row = {
            "Sample": sample["sample_number"],
            "Source frame": sample["frame_index"],
            "Face detected": "No",
            "Fake score": np.nan,
        }

        if sample["read_success"]:
            crop_bgr = crop_largest_face(sample["frame_bgr"])
            if crop_bgr is not None:
                row["Face detected"] = "Yes"
                valid_crops.append(crop_bgr)
                valid_row_indices.append(row_index)
                crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
                gallery_items.append(
                    (Image.fromarray(crop_rgb), f"Sample {sample['sample_number']} · frame {sample['frame_index']}")
                )

        rows.append(row)

    if not valid_crops:
        raise gr.Error(
            "No valid face was detected in the sampled frames. "
            "Try a video containing a larger, unobstructed frontal face."
        )

    scores = predict_fake_scores(valid_crops)
    for row_index, score in zip(valid_row_indices, scores):
        rows[row_index]["Fake score"] = float(score)

    mean_score = float(np.mean(scores))
    standard_label = label_for_score(mean_score, STANDARD_THRESHOLD)
    sensitive_label = label_for_score(mean_score, SENSITIVE_THRESHOLD)

    summary = (
        "## Video-analysis result\n\n"
        f"**Mean E3b video score:** `{mean_score:.4f}`  \n"
        f"**Standard mode ({STANDARD_THRESHOLD:.2f}):** {result_badge(standard_label)}  \n"
        f"**High-sensitivity mode ({SENSITIVE_THRESHOLD:.2f}):** {result_badge(sensitive_label)}  \n"
        f"**Successfully analysed faces:** {len(scores)} of {len(sampled_frames)} sampled frames  \n"
        f"**Reported source-video frames:** {total_frames}\n\n"
        "> This score is a model output, not a calibrated probability or conclusive forensic finding. "
        "Cross-dataset evaluation demonstrated sensitivity to domain shift."
    )

    score_table = pd.DataFrame(rows)
    figure, axis = plt.subplots(figsize=(9, 4.5))
    analysed_samples = [rows[index]["Sample"] for index in valid_row_indices]
    axis.plot(analysed_samples, scores, marker="o", linewidth=2, label="Frame fake score")
    axis.axhline(STANDARD_THRESHOLD, color="grey", linestyle="--", label="Standard threshold 0.50")
    axis.axhline(SENSITIVE_THRESHOLD, color="red", linestyle="--", label="High-sensitivity threshold 0.17")
    axis.axhline(mean_score, color="navy", linestyle=":", label=f"Mean score {mean_score:.4f}")
    axis.set_ylim(0, 1)
    axis.set_xlabel("Sampled frame number")
    axis.set_ylabel("E3b fake score")
    axis.set_title("Frame-level E3b scores for the uploaded video")
    axis.grid(True, alpha=0.3)
    axis.legend(loc="best")
    figure.tight_layout()

    return (
        summary,
        mean_score,
        explorer_message(mean_score, exploratory_threshold),
        score_table,
        figure,
        gallery_items,
        mean_score,
    )

## 6. Launch the Gradio application

If `CREATE_PUBLIC_LINK=True`, Colab creates a temporary public Gradio URL. Do not upload confidential or personally sensitive videos to a publicly reachable demonstration session.

In [ ]:
with gr.Blocks(title="E3b Deepfake-Detection Research Prototype") as demo:
    gr.Markdown(
        "# E3b Deepfake-Detection Research Prototype\n"
        "Upload one facial video for frame sampling, RetinaFace preprocessing and E3b analysis. "
        "This is an MSc research demonstrator, not a production forensic service."
    )

    stored_video_score = gr.State(value=None)

    with gr.Tab("Analyse video"):
        video_input = gr.File(
            label="Select a video file",
            file_types=["video"],
            type="filepath",
        )
        analyse_button = gr.Button("Analyse video", variant="primary")
        result_output = gr.Markdown("Upload a video and select **Analyse video**.")
        mean_score_output = gr.Number(label="Mean E3b video score", precision=4)

        with gr.Row():
            score_plot_output = gr.Plot(label="Frame-score plot")
            face_gallery_output = gr.Gallery(
                label="Detected face crops",
                columns=5,
                rows=2,
                height="auto",
            )

        score_table_output = gr.Dataframe(
            headers=["Sample", "Source frame", "Face detected", "Fake score"],
            label="Sampled-frame analysis",
            interactive=False,
        )

    with gr.Tab("Threshold explorer"):
        gr.Markdown(
            "Move the threshold to see how the **already calculated** video score is converted "
            "into a label. Moving this control does not rerun E3b. Only 0.50 and 0.17 are "
            "predefined study operating points."
        )
        threshold_slider = gr.Slider(
            minimum=0.01,
            maximum=0.99,
            value=SENSITIVE_THRESHOLD,
            step=0.01,
            label="Exploratory video decision threshold",
        )
        explorer_output = gr.Markdown(
            "Upload and analyse a video before using the threshold explorer."
        )

    analyse_button.click(
        fn=analyse_video,
        inputs=[video_input, threshold_slider],
        outputs=[
            result_output,
            mean_score_output,
            explorer_output,
            score_table_output,
            score_plot_output,
            face_gallery_output,
            stored_video_score,
        ],
    )

    threshold_slider.change(
        fn=explorer_message,
        inputs=[stored_video_score, threshold_slider],
        outputs=explorer_output,
    )

demo.queue()
demo.launch(share=CREATE_PUBLIC_LINK, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://06a6afee0ddb449620.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


26-09-02 11:57:29 - Directory /root/.deepface created
26-09-02 11:57:29 - Directory /root/.deepface/weights created
26-09-02 11:57:29 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5
100%|██████████| 119M/119M [00:01<00:00, 95.8MB/s]


## Implementation safeguards

- The application loads the retained E3b checkpoint and does not retrain it.
- E3b input processing remains $224\times224$, RGB, float32 scaling to $[0,1]$, without ImageNet normalisation.
- Output index 1 is interpreted as the fake score, consistent with the E3b checkpoint.
- Ten frames are sampled evenly across the uploaded video where possible.
- The largest RetinaFace detection is expanded by 15% before resizing.
- Video inference uses the mean fake score across frames with valid face detections.
- The values 0.50 and 0.17 are fixed study operating points.
- Slider values are exploratory and do not constitute target-domain calibration.
- The output is presented as decision support, not conclusive forensic evidence.